In [21]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('./scripts')  
import preprocesamiento
import feature_engineering
import model_lgb
importlib.reload(preprocesamiento)
importlib.reload(model_lgb)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

# Experimento 7: 
- LGBM
- Estandarizacion del target
- Usando funcion entrenamiento: semillerio_en_prediccion
- Mismas variables
- Pesos: (log o max?)
- sqlite:///optuna_studies_v16.db
- Kaggle =  0.30


##### Levantamos el dataset con target ya calculado

In [2]:
df = pd.read_csv("./datasets/periodo_x_producto_con_target_transformado_201912.csv", sep=',', encoding='utf-8')
print("Dataset sin transformar tenia esto: (31362, 19)")
df.shape

Dataset sin transformar tenia esto: (31362, 19)


(31362, 35)

In [3]:
columnas_baseline = df.columns.tolist()
columnas_baseline

['product_id',
 'periodo',
 'nacimiento_producto',
 'muerte_producto',
 'mes_n',
 'total_meses',
 'producto_nuevo',
 'ciclo_de_vida_inicial',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'stock_final',
 'tn',
 'plan_precios_cuidados',
 'cust_request_qty',
 'cust_request_tn',
 'target',
 'tn_mean',
 'tn_std',
 'tn_zscore',
 'stock_final_mean',
 'stock_final_std',
 'stock_final_zscore',
 'cust_request_qty_mean',
 'cust_request_qty_std',
 'cust_request_qty_zscore',
 'cust_request_tn_mean',
 'cust_request_tn_std',
 'cust_request_tn_zscore',
 'tn_log',
 'stock_final_log',
 'cust_request_qty_log',
 'cust_request_tn_log']

##### Preprocesamiento a la minima expresión :)

In [4]:
df = feature_engineering.create_category_features_cat1(df)
df = feature_engineering.create_category_features_cat2(df)
df = feature_engineering.create_category_features_cat3(df)

In [5]:
# ##### aplicamos OHE
df = preprocesamiento.aplicarOHE(df)
df.shape

(31362, 187)

### Feature Engineering

##### Neural Prophet

In [6]:
neural_prophet_fe = pd.read_csv("./datasets/features_neuralprophet_completo.csv", sep=',', encoding='utf-8')
neural_prophet_fe['ds'] = pd.to_datetime(neural_prophet_fe['ds'], errors='coerce')
# Versión alternativa más robusta:
neural_prophet_fe['periodo'] = neural_prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
neural_prophet_fe = neural_prophet_fe[['periodo', 'product_id', 'trend', "season_yearly", "season_monthly"]]
df = df.merge(neural_prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 190)

##### Prophet

In [7]:
prophet_fe = pd.read_csv("./datasets/prophet_features_tn_zscore.csv", sep=',', encoding='utf-8')
prophet_fe['ds'] = pd.to_datetime(prophet_fe['ds'], errors='coerce')
prophet_fe['periodo'] = prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
prophet_fe = prophet_fe[['periodo', 'product_id', 'trend_add', "yearly_add", "additive_terms", 'trend_mult', 'yearly_mult', 'multiplicative_terms']]
df = df.merge(prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 196)

##### FE Moviles

In [8]:
df = feature_engineering.get_lags(df, "tn", 201912)
df = feature_engineering.get_delta_lags(df, "tn", 24)
df = feature_engineering.get_rolling_means(df, "tn", 201912)
df = feature_engineering.get_rolling_stds(df, "tn", 201912)
df = feature_engineering.get_rolling_mins(df, "tn", 201912)
df = feature_engineering.get_rolling_maxs(df, "tn", 201912)
df = feature_engineering.get_rolling_medians(df, "tn", 201912)
df = feature_engineering.get_rolling_skewness(df, "tn", 201912)
df = feature_engineering.get_autocorrelaciones(df, "tn", 201912)
df.shape

(31362, 758)

In [9]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)
df.shape

(31362, 1213)

In [10]:
df = feature_engineering.get_lags(df, "stock_final", 201912)
df = feature_engineering.get_delta_lags(df, "stock_final", 24)
df = feature_engineering.get_rolling_means(df, "stock_final", 201912)
df = feature_engineering.get_rolling_stds(df, "stock_final", 201912)
df = feature_engineering.get_rolling_mins(df, "stock_final", 201912)
df = feature_engineering.get_rolling_maxs(df, "stock_final", 201912)
df.shape

(31362, 1668)

Features Diana

In [ ]:
df = feature_engineering.calcular_diferencia_con_medias_moviles(df)
df = feature_engineering.calcular_ratios_con_medias_moviles(df)
df.shape

(31362, 1704)

##### FE Moviles sobre otras variables

In [56]:
# #  stock final
# df = feature_engineering.get_lagsEspecificos(df, col='stock_final_zscore')
# df = feature_engineering.get_delta_lags_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_means_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_stds_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_medians_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_mins_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='stock_final_zscore')

#  cust_request_qty
# df = feature_engineering.get_lagsEspecificos(df, col='cust_request_qty')
# df = feature_engineering.get_delta_lags_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_means_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_stds_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_mins_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_medians_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='cust_request_qty')

##### FE Calendario

In [12]:
df = feature_engineering.generar_ids(df)
df = feature_engineering.get_componentesTemporales(df)
df = feature_engineering.get_anomaliasPoliticas(df)
# df = feature_engineering.descomposicion_serie_temporal(df, col='tn')
df.shape

(31362, 1729)

##### FE sobre FE

In [13]:
df = feature_engineering.chatGPT_features_serie(df, "tn")
df = feature_engineering.mes_con_feriado(df)
df.shape

(31362, 1758)

##### Variables Exogenas

In [14]:
df = feature_engineering.get_dolar(df)
df = feature_engineering.get_IPC(df)
df['ipc'] = df['ipc'].str.replace(',', '.').astype(float)
df['dolar'] = df['dolar'].str.replace(',', '.').astype(float)
# df.drop(columns=['ds'], inplace=True)
df.fillna(0, inplace=True) ##### EXPERIMENTAR
df = feature_engineering.correlacion_exogenas(df)
df = feature_engineering.get_mes_receso_escolar(df)
df.shape

(31362, 1761)

##### Nuevas FE

In [15]:
df = feature_engineering.create_ratio_features(df)
df = feature_engineering.enhance_lifecycle_features(df)
# df = feature_engineering.create_category_features(df)
df = feature_engineering.create_regime_features(df)
df = feature_engineering.create_nonlinear_trends(df)
df = feature_engineering.create_temporal_interactions(df)
df = feature_engineering.create_asymmetric_window_features(df)
df = feature_engineering.recomendaciones_deepseek(df)
df = feature_engineering.get_nuevas_features(df)
df.shape

(31362, 1798)

##### Elimino aquellas que no sirven

In [ ]:
import json
import pandas as pd
import csv

with open("./feature_importance/v19.json") as f:
    data = json.load(f)

# Crear una lista de tuplas (feature, value)
features_values = [(feature, value) for feature, value in data.items()]

# Guardar en un archivo CSV
with open('./feature_importance/v19.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['feature', 'importance'])  # Escribir el encabezado
    writer.writerows(features_values)      # Escribir los datos

print("Archivo CSV generado exitosamente: features_values.csv")

Archivo CSV generado exitosamente: features_values.csv


In [16]:
importantes = pd.read_csv("./feature_importance/v19.csv", sep=',', encoding='utf-8')
no_importantes = importantes[importantes['importance'] == 0]
no_importantes = no_importantes[~no_importantes['feature'].isin(columnas_baseline)]
no_importantes

,feature,importance
1103,tn_rolling_std_20,0.0
1104,tn_rolling_std_22,0.0
1105,tn_rolling_std_25,0.0
1106,tn_rolling_std_26,0.0
1107,tn_rolling_std_27,0.0
...,...,...
1781,cat3_Acond Bebe,0.0
1782,tn_rolling_median_25,0.0
1783,tn_rolling_median_24,0.0
1786,tn_rolling_std_1,0.0


In [17]:
cols_a_eliminar = no_importantes.feature.unique()
print(f"Antes de eliminar: {df.shape[1]} columnas")
df = df.drop(columns=cols_a_eliminar, errors='ignore')
print(f"Después de eliminar: {df.shape[1]} columnas")

Antes de eliminar: 1798 columnas
Después de eliminar: 1116 columnas


Eliminar object/categorical columnas

In [18]:
df = df.select_dtypes(exclude=['datetime', 'datetime64', 'object'])

Train Test Split

In [19]:
train = df[df['periodo'] <= 201912]
test = df[df['periodo'] == 201912]

Entrenamiento

In [20]:
model_lgb.optimizar_con_optuna_con_semillerio_db(train, version="v20", n_trials=500)


Para visualizar los resultados en tiempo real:
1. Abre otra terminal y ejecuta:
   optuna-dashboard sqlite:///optuna_studies_v20.db
2. Abre en tu navegador: http://127.0.0.1:8080/


[I 2025-07-09 10:16:20,395] Using an existing study with name 'lightgbm_optimization_v20' instead of creating a new one.
[I 2025-07-09 10:24:27,886] Trial 120 finished with value: 0.33345932270869916 and parameters: {'num_leaves': 73, 'learning_rate': 0.24407684875617658, 'feature_fraction': 0.8237624154096727, 'bagging_fraction': 0.9776262944059598, 'bagging_freq': 5, 'lambda_l1': 0.000867696677065586, 'lambda_l2': 3.1753580321836803e-06, 'min_child_samples': 35, 'max_depth': 10, 'max_bin': 461, 'min_data_in_leaf': 98, 'extra_trees': False, 'early_stopping_rounds': 16, 'path_smooth': 0.01855501975285532, 'min_gain_to_split': 0.059906632092850645}. Best is trial 91 with value: 0.05452697337146212.


Mejor trial hasta ahora: RMSE=0.054527, Parámetros={'num_leaves': 65, 'learning_rate': 0.21216746368001035, 'feature_fraction': 0.9781407572059055, 'bagging_fraction': 0.9911984609582816, 'bagging_freq': 6, 'lambda_l1': 0.014755364659435312, 'lambda_l2': 7.391699873589852e-06, 'min_child_samples': 31, 'max_depth': 10, 'max_bin': 481, 'min_data_in_leaf': 34, 'extra_trees': False, 'early_stopping_rounds': 23, 'path_smooth': 0.0392891614991775, 'min_gain_to_split': 0.02082318088761467}


[I 2025-07-09 10:38:27,200] Trial 121 finished with value: 0.1239903318460078 and parameters: {'num_leaves': 66, 'learning_rate': 0.22514439322776278, 'feature_fraction': 0.999781789698398, 'bagging_fraction': 0.9919476580835159, 'bagging_freq': 6, 'lambda_l1': 1.2396414771128246e-07, 'lambda_l2': 1.1555691800459419e-05, 'min_child_samples': 29, 'max_depth': 9, 'max_bin': 472, 'min_data_in_leaf': 46, 'extra_trees': False, 'early_stopping_rounds': 19, 'path_smooth': 0.06982189279718097, 'min_gain_to_split': 0.022361415841808216}. Best is trial 91 with value: 0.05452697337146212.


Mejor trial hasta ahora: RMSE=0.054527, Parámetros={'num_leaves': 65, 'learning_rate': 0.21216746368001035, 'feature_fraction': 0.9781407572059055, 'bagging_fraction': 0.9911984609582816, 'bagging_freq': 6, 'lambda_l1': 0.014755364659435312, 'lambda_l2': 7.391699873589852e-06, 'min_child_samples': 31, 'max_depth': 10, 'max_bin': 481, 'min_data_in_leaf': 34, 'extra_trees': False, 'early_stopping_rounds': 23, 'path_smooth': 0.0392891614991775, 'min_gain_to_split': 0.02082318088761467}


...

[I 2025-07-12 10:16:07,313] Trial 420 finished with value: 0.29201634137121574 and parameters: {'num_leaves': 82, 'learning_rate': 0.227143349237452, 'feature_fraction': 0.9160951960154565, 'bagging_fraction': 0.9761069102212239, 'bagging_freq': 10, 'lambda_l1': 1.055328993912688e-05, 'lambda_l2': 7.199872423617586e-06, 'min_child_samples': 30, 'max_depth': 10, 'max_bin': 471, 'min_data_in_leaf': 27, 'extra_trees': False, 'early_stopping_rounds': 28, 'path_smooth': 0.9534435639864683, 'min_gain_to_split': 0.0172848217320848}. Best is trial 371 with value: 0.01917960973943581.


Mejor trial hasta ahora: RMSE=0.019180, Parámetros={'num_leaves': 81, 'learning_rate': 0.2067657403659724, 'feature_fraction': 0.897740362101773, 'bagging_fraction': 0.9783984895510337, 'bagging_freq': 5, 'lambda_l1': 6.437768634924736e-05, 'lambda_l2': 4.723296621845035e-06, 'min_child_samples': 29, 'max_depth': 10, 'max_bin': 465, 'min_data_in_leaf': 28, 'extra_trees': False, 'early_stopping_rounds': 24, 'path_smooth': 0.9836390817928878, 'min_gain_to_split': 0.04278735382396155}


[I 2025-07-12 10:49:26,854] Trial 421 finished with value: 0.49700976781221307 and parameters: {'num_leaves': 85, 'learning_rate': 0.04737111817811871, 'feature_fraction': 0.9730638296736132, 'bagging_fraction': 0.9869199190119768, 'bagging_freq': 5, 'lambda_l1': 1.4251696452175481e-05, 'lambda_l2': 1.869104692989137e-06, 'min_child_samples': 31, 'max_depth': 10, 'max_bin': 492, 'min_data_in_leaf': 34, 'extra_trees': False, 'early_stopping_rounds': 25, 'path_smooth': 0.9343715304632452, 'min_gain_to_split': 0.030274242687886182}. Best is trial 371 with value: 0.01917960973943581.


Mejor trial hasta ahora: RMSE=0.019180, Parámetros={'num_leaves': 81, 'learning_rate': 0.2067657403659724, 'feature_fraction': 0.897740362101773, 'bagging_fraction': 0.9783984895510337, 'bagging_freq': 5, 'lambda_l1': 6.437768634924736e-05, 'lambda_l2': 4.723296621845035e-06, 'min_child_samples': 29, 'max_depth': 10, 'max_bin': 465, 'min_data_in_leaf': 28, 'extra_trees': False, 'early_stopping_rounds': 24, 'path_smooth': 0.9836390817928878, 'min_gain_to_split': 0.04278735382396155}
Estudio guardado en: sqlite:///optuna_studies_v20.db

Mejores hiperparámetros encontrados:
num_leaves: 81
learning_rate: 0.2067657403659724
feature_fraction: 0.897740362101773
bagging_fraction: 0.9783984895510337
bagging_freq: 5
lambda_l1: 6.437768634924736e-05
lambda_l2: 4.723296621845035e-06
min_child_samples: 29
max_depth: 10
max_bin: 465
min_data_in_leaf: 28
extra_trees: False
early_stopping_rounds: 24
path_smooth: 0.9836390817928878
min_gain_to_split: 0.04278735382396155


(<optuna.study.study.Study at 0x27185e765d0>,
 {'num_leaves': 81,
  'learning_rate': 0.2067657403659724,
  'feature_fraction': 0.897740362101773,
  'bagging_fraction': 0.9783984895510337,
  'bagging_freq': 5,
  'lambda_l1': 6.437768634924736e-05,
  'lambda_l2': 4.723296621845035e-06,
  'min_child_samples': 29,
  'max_depth': 10,
  'max_bin': 465,
  'min_data_in_leaf': 28,
  'extra_trees': False,
  'early_stopping_rounds': 24,
  'path_smooth': 0.9836390817928878,
  'min_gain_to_split': 0.04278735382396155,
  'objective': 'regression',
  'metric': 'rmse',
  'boosting_type': 'gbdt',
  'verbosity': -1})

Prediccion

In [22]:
df_future = model_lgb.semillerio_en_prediccion_con_pesos(train, test, version="v20")

In [23]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.492389
30477,201912,20002,0.0,0.257005
30478,201912,20003,0.0,-0.084902
30479,201912,20004,0.0,0.067027
30480,201912,20005,0.0,0.520183
...,...,...,...,...
31357,201912,21265,0.0,0.055806
31358,201912,21266,0.0,0.002210
31359,201912,21267,0.0,-0.173355
31360,201912,21271,0.0,0.038051


Filtramos los 180 productos

In [24]:
productos_ok = pd.read_csv("https://storage.googleapis.com/open-courses/austral2025-af91/labo3v/product_id_apredecir201912.txt", sep="\t")
df_future = df_future[df_future['periodo'] == 201912]
df_future = df_future[df_future['product_id'].isin(productos_ok['product_id'].unique())]


In [25]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.492389
30477,201912,20002,0.0,0.257005
30478,201912,20003,0.0,-0.084902
30479,201912,20004,0.0,0.067027
30480,201912,20005,0.0,0.520183
...,...,...,...,...
31355,201912,21263,0.0,-0.171320
31357,201912,21265,0.0,0.055806
31358,201912,21266,0.0,0.002210
31359,201912,21267,0.0,-0.173355


In [26]:
df_future_copy = df_future.copy()

In [27]:
import os
ruta_archivo = f'./datasets/tn_stats_201912.csv'
    
df_stats = pd.DataFrame()

if os.path.exists(ruta_archivo) and ruta_archivo.endswith('.csv'):
    df_stats = pd.read_csv(ruta_archivo, sep=',')

df_stats

,product_id,tn_mean,tn_std
0,20001,1398.344322,293.975388
1,20002,1009.368178,299.585187
2,20003,889.004243,287.951952
3,20004,671.615383,221.310769
4,20005,644.200514,215.220300
...,...,...,...
1159,21271,0.024268,0.019484
1160,21273,0.057242,0.124272
1161,21274,0.067028,0.096980
1162,21276,0.045447,0.041380


In [28]:
df_future_copy = df_future_copy.merge(df_stats, on=['product_id'], how='left')
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std
0,201912,20001,0.0,0.492389,1398.344322,293.975388
1,201912,20002,0.0,0.257005,1009.368178,299.585187
2,201912,20003,0.0,-0.084902,889.004243,287.951952
3,201912,20004,0.0,0.067027,671.615383,221.310769
4,201912,20005,0.0,0.520183,644.200514,215.220300
...,...,...,...,...,...,...
775,201912,21263,0.0,-0.171320,0.089233,0.148180
776,201912,21265,0.0,0.055806,0.089541,0.103219
777,201912,21266,0.0,0.002210,0.094659,0.100530
778,201912,21267,0.0,-0.173355,0.092835,0.075836


In [29]:
df_future_copy['tn'] = df_future_copy['pred'] * df_future_copy['tn_std'] + df_future_copy['tn_mean']
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std,tn
0,201912,20001,0.0,0.492389,1398.344322,293.975388,1543.094708
1,201912,20002,0.0,0.257005,1009.368178,299.585187,1086.362960
2,201912,20003,0.0,-0.084902,889.004243,287.951952,864.556542
3,201912,20004,0.0,0.067027,671.615383,221.310769,686.449089
4,201912,20005,0.0,0.520183,644.200514,215.220300,756.154405
...,...,...,...,...,...,...,...
775,201912,21263,0.0,-0.171320,0.089233,0.148180,0.063847
776,201912,21265,0.0,0.055806,0.089541,0.103219,0.095301
777,201912,21266,0.0,0.002210,0.094659,0.100530,0.094881
778,201912,21267,0.0,-0.173355,0.092835,0.075836,0.079688


Vemos cuantos negativos hay

In [30]:
df_future_copy[df_future_copy['tn'] < 0]

,periodo,product_id,target,pred,tn_mean,tn_std,tn


Reemplazamos los negativos por el promedio de ultimos 12 meses

In [ ]:
# promedio780 = model_lgb.promedio_12_meses_780p()
# df_future = df_future.merge(promedio780, on='product_id', how='left')
# df_future.drop(columns=['target','periodo'], inplace=True)
# df_future.loc[df_future['pred'] < 0, 'pred'] = df_future['tn']
# df_future



,product_id,pred,tn
0,20001,1397.305481,1454.732720
1,20002,1086.538942,1175.437142
2,20003,747.163659,784.976407
3,20004,565.799872,627.215328
4,20005,638.965713,668.270104
...,...,...,...
775,21263,0.029993,0.029993
776,21265,0.791975,0.089541
777,21266,0.094659,0.094659
778,21267,0.092835,0.092835


Guardamos el archivo

In [31]:
# df_future_copy.drop(columns=['tn'], inplace=True)
# df_future_copy.rename(columns={'pred': 'tn'}, inplace=True)
df_future_copy[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v8_semicompleto.csv", index=False, sep=',')

Ensemble

In [32]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl']) / 2
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v8_ensemble_v2.csv", index=False, sep=',')

In [33]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ag = pd.read_csv("./outputs/prediccion_autogluon_2ventanas.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_ag'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble = df_ensemble.merge(df_ag, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl'] + df_ensemble['tn_ag']) / 3
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v8_ensemble_3models_semicompleto.csv", index=False, sep=',')